In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [19]:
raw = spark.read.csv("hdfs://namenode:9000/data/raw/accidentes/", header=True, inferSchema=True)
raw.write.mode("overwrite").parquet("hdfs://namenode:9000/data/staging/accidentes")

In [20]:
raw.count()

7728394

In [17]:
stg = spark.read.parquet("hdfs://namenode:9000/data/staging/accidentes/")

In [42]:
stg.count()

7728394

In [43]:
stg.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'End_Lat',
 'End_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Zipcode',
 'Country',
 'Timezone',
 'Airport_Code',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [44]:
from pyspark.sql import functions as F

df_counts = stg.groupBy("ID").count().filter("count > 1")
print("Cantidad de IDs que tienen duplicados:", df_counts.count())

df_counts.select("ID").show(10, truncate=False)

Cantidad de IDs que tienen duplicados: 0
+---+
|ID |
+---+
+---+



In [18]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = stg.count()
cols = [c for c in stg.columns if c != "ID"]
tipos = dict(stg.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.col(col_nombre).isNull()| F.isnan(F.col(col_nombre))
    elif tipos[col_nombre] in ("timestamp", "boolean"):
        condicion = F.col(col_nombre).isNull()
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto) | F.col(col_nombre).isNull()
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = stg.agg(*exprs).collect()[0].asDict()

In [5]:
for c in cols:
    n = conteos[c] or 0
    print(f"Columna: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")

Columna: Source | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Severity | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lat | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Start_Lng | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: End_Lat | Nulos/NaN: 3402762 | Porcentaje: 44.03%
Columna: End_Lng | Nulos/NaN: 3402762 | Porcentaje: 44.03%
Columna: Distance(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Description | Nulos/NaN: 5 | Porcentaje: 0.00%
Columna: Street | Nulos/NaN: 10869 | Porcentaje: 0.14%
Columna: City | Nulos/NaN: 253 | Porcentaje: 0.00%
Columna: County | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: State | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Zipcode | Nulos/NaN: 1915 | Porcentaje: 0.02%
Columna: Country | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna: Timezone | Nulos/NaN: 7808 | Porcentaje: 0.10%
Columna: Airport_Code | Nulos/NaN: 22635 | Porcentaje: 0.29%
Columna: Weather_T

In [19]:
stg= stg.drop("End_Lat", "End_Lng","Airport_Code","Zipcode","Country")

In [21]:
total_antes = stg.count()

stg = stg.dropna()

total_despues = stg.count()
total_eliminados = total_antes - total_despues

print(f"Registros originales: {total_antes:,}")
print(f"Registros limpios finales en accidents: {total_despues:,}")
print(f"Total de filas eliminadas: {total_eliminados:,}")

Registros originales: 7,728,394
Registros limpios finales en accidents: 5,208,003
Total de filas eliminadas: 2,520,391


In [9]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType
exprs = []
total_filas = stg.count()
cols = [c for c in stg.columns if c != "ID"]
tipos = dict(stg.dtypes)
valores_nulos_texto = ["", "NaN", "nan", "None", "N/A", "NULL"]
for col_nombre in cols:
    if tipos[col_nombre] in ("double", "float"):
        condicion = F.col(col_nombre).isNull()| F.isnan(F.col(col_nombre))
    elif tipos[col_nombre] in ("timestamp", "boolean"):
        condicion = F.col(col_nombre).isNull()
    else: 
        condicion = F.trim(F.col(col_nombre)).isin(valores_nulos_texto) | F.col(col_nombre).isNull()
    exprs.append(F.sum(F.when(condicion, 1).otherwise(0)).alias(col_nombre))
conteos = stg.agg(*exprs).collect()[0].asDict()

In [11]:
i = 1
for c in cols:
    n = conteos[c] or 0
    print(f"Columna{i}: {c} | Nulos/NaN: {n} | Porcentaje: {n / total_filas * 100:.2f}%")
    i+=1

Columna1: Source | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna2: Severity | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna3: Start_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna4: End_Time | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna5: Start_Lat | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna6: Start_Lng | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna7: Distance(mi) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna8: Description | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna9: Street | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna10: City | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna11: County | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna12: State | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna13: Timezone | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna14: Weather_Timestamp | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna15: Temperature(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna16: Wind_Chill(F) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna17: Humidity(%) | Nulos/NaN: 0 | Porcentaje: 0.00%
Columna18: Pressure(in) | Nulos/NaN: 0 | Porcentaje: 

In [22]:
import re
def normalizar(n):
    return re.sub(r"[^a-z0-9]+", "_", n.strip().lower()).strip("_")

In [9]:
stg.columns

['ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'Distance(mi)',
 'Description',
 'Street',
 'City',
 'County',
 'State',
 'Timezone',
 'Weather_Timestamp',
 'Temperature(F)',
 'Wind_Chill(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Direction',
 'Wind_Speed(mph)',
 'Precipitation(in)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Civil_Twilight',
 'Nautical_Twilight',
 'Astronomical_Twilight']

In [23]:
base = stg.toDF(*[normalizar(c) for c in stg.columns])
base.columns

['id',
 'source',
 'severity',
 'start_time',
 'end_time',
 'start_lat',
 'start_lng',
 'distance_mi',
 'description',
 'street',
 'city',
 'county',
 'state',
 'timezone',
 'weather_timestamp',
 'temperature_f',
 'wind_chill_f',
 'humidity',
 'pressure_in',
 'visibility_mi',
 'wind_direction',
 'wind_speed_mph',
 'precipitation_in',
 'weather_condition',
 'amenity',
 'bump',
 'crossing',
 'give_way',
 'junction',
 'no_exit',
 'railway',
 'roundabout',
 'station',
 'stop',
 'traffic_calming',
 'traffic_signal',
 'turning_loop',
 'sunrise_sunset',
 'civil_twilight',
 'nautical_twilight',
 'astronomical_twilight']

In [24]:
cols_accidents = [c for c in (base.columns[0:14] + base.columns[24:])]
cols_weather = [c for c in (base.columns[14:24])]

In [25]:
accidents = base.select(cols_accidents)

weather = base.select("ID",*cols_weather)

weather.show(1)

accidents.show(1)

+---------+-------------------+-------------+------------+--------+-----------+-------------+--------------+--------------+----------------+-----------------+
|       ID|  weather_timestamp|temperature_f|wind_chill_f|humidity|pressure_in|visibility_mi|wind_direction|wind_speed_mph|precipitation_in|weather_condition|
+---------+-------------------+-------------+------------+--------+-----------+-------------+--------------+--------------+----------------+-----------------+
|A-5199312|2022-10-07 23:54:00|         79.0|        79.0|    34.0|       29.5|          8.0|             E|           5.0|             0.0|             Fair|
+---------+-------------------+-------------+------------+--------+-----------+-------------+--------------+--------------+----------------+-----------------+
only showing top 1 row
+---------+-------+--------+-------------------+-------------------+---------+-----------+-----------+--------------------+---------+-----------+------+-----+----------+-------+-----

In [14]:
accidents.printSchema()  

root
 |-- id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- severity: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- distance_mi: double (nullable = true)
 |-- description: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- amenity: boolean (nullable = true)
 |-- bump: boolean (nullable = true)
 |-- crossing: boolean (nullable = true)
 |-- give_way: boolean (nullable = true)
 |-- junction: boolean (nullable = true)
 |-- no_exit: boolean (nullable = true)
 |-- railway: boolean (nullable = true)
 |-- roundabout: boolean (nullable = true)
 |-- station: boolean (nullable = true)
 |-- stop: boolean (nullable = true)
 |-- traffic_calming: boolean (nullab

In [2]:
OUT = "hdfs://namenode:9000/data/processed/accidentes"
accidents.write.mode("overwrite").parquet(f"{OUT}/accidents")
weather.write.mode("overwrite").parquet(f"{OUT}/weather")

NameError: name 'accidents' is not defined

In [3]:
PG_URL = "jdbc:postgresql://postgres:5432/smartcities?rewriteBatchedInserts=true"
PG_PROPS = {"user": "postgres", "password": "postgres", "driver": "org.postgresql.Driver"}

In [7]:
from pyspark.sql import Window
from pyspark.sql import functions as F

In [5]:
acc = spark.read.parquet(f"{OUT}/accidents")
ROAD = ["amenity","bump","crossing","give_way","junction","no_exit","railway",
        "roundabout","station","stop","traffic_calming","traffic_signal","turning_loop"]

def con_id(df, cols, id_col):
    return df.select(*cols).distinct().withColumn(id_col, F.row_number().over(Window.orderBy(*cols)))

In [8]:
sources = con_id(acc, ["source"], "source_id")
cities  = con_id(acc, ["city", "county", "state"], "city_id")
states  = acc.select(F.col("state").alias("state_code")).distinct()

In [9]:
fk = acc.join(sources, "source").join(cities, ["city", "county", "state"])

In [10]:
accidents_tbl = fk.select("id", "source_id", "severity", "start_time", "end_time", "start_lat",
    "start_lng", "distance_mi", "description", "street", "city_id", "timezone",
    "sunrise_sunset", "civil_twilight", "nautical_twilight", "astronomical_twilight")


In [11]:
road_tbl = fk.select("id", *ROAD)
cities_tbl = cities.select("city_id", "city", "county", F.col("state").alias("state_code"))

In [12]:
def a_pg(df, tabla):
    df.write.option("batchsize", 10000).jdbc(PG_URL, tabla, mode="append", properties=PG_PROPS)

In [13]:
a_pg(sources, "sources")
a_pg(states, "states")
a_pg(cities_tbl, "cities")
a_pg(accidents_tbl.repartition(8), "accidents")
a_pg(road_tbl.repartition(8), "accident_road_features")

c:\Users\steve\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\connect\expressions.py:1091: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
